# 08 · Master Pipeline Kaggle

Pipeline completo: prepara scripts, detecta GPU, instala/atualiza ComfyUI + custom nodes, sincroniza modelos do Kaggle Dataset → SSD, configura output no Google Drive, inicia ComfyUI com health check.


In [ ]:
from pathlib import Path
import subprocess, sys, os, time, json

REPO_URL = "https://github.com/automadevs/colab-pipeline.git"
WORKDIR = Path("/kaggle/working")
SCRIPTS_DIR = WORKDIR / "scripts"
COMFYUI_DIR = WORKDIR / "ComfyUI"
MODELS_DIR = COMFYUI_DIR / "models"
DATASET = "automamermaid/comfydocs"
DRIVE_BASE = "Automa/ComfyUI"

print("GPU / DISCO")
subprocess.run(["nvidia-smi"], check=False)
subprocess.run(["df", "-h", str(WORKDIR)], check=False)


In [ ]:
# Baixa/atualiza somente o código do projeto. Nenhuma credencial é necessária.
if (SCRIPTS_DIR / "kaggle_sync.py").exists():
    subprocess.run(["git", "-C", str(WORKDIR / "colab-pipeline"), "pull"], check=False)
else:
    repo_dir = WORKDIR / "colab-pipeline"
    if repo_dir.exists():
        subprocess.run(["git", "-C", str(repo_dir), "pull"], check=False)
    else:
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(repo_dir)], check=True)
    import shutil
    if SCRIPTS_DIR.exists(): shutil.rmtree(SCRIPTS_DIR)
    shutil.copytree(repo_dir / "scripts", SCRIPTS_DIR)

sys.path.insert(0, str(SCRIPTS_DIR))
print("Scripts disponíveis:", sorted(p.name for p in SCRIPTS_DIR.glob("*.py")))

In [ ]:
# GPU real do Notebook
from gpu_detect import detect_gpu
GPU_INFO = detect_gpu()
print(json.dumps(GPU_INFO, indent=2, ensure_ascii=False))
if not GPU_INFO.get("has_gpu"):
    raise RuntimeError("GPU NVIDIA não detectada. Ative Accelerator → GPU no Kaggle antes de continuar.")

In [ ]:
# Configurar Google Drive (rclone + service account)
from kaggle_drive_sync import get_drive_path, setup_rclone_kaggle

print("=" * 60)
print("CONFIGURANDO GOOGLE DRIVE")
print("=" * 60)

try:
    drive_path = get_drive_path(drive_base=DRIVE_BASE, env="kaggle")
    print(f"[INFO] Google Drive montado em: {drive_path}")
    
    # Criar estrutura de pastas
    for subdir in ["outputs", "workflows", "logs", "metadata"]:
        (drive_path / subdir).mkdir(parents=True, exist_ok=True)
    
    OUTPUT_DIR = drive_path / "outputs"
    WORKFLOWS_DIR = drive_path / "workflows"
    LOGS_DIR = drive_path / "logs"
    
    print(f"[INFO] Output dir: {OUTPUT_DIR}")
    print(f"[INFO] Workflows dir: {WORKFLOWS_DIR}")
    print(f"[INFO] Logs dir: {LOGS_DIR}")
    
except Exception as e:
    print(f"[WARN] Falha ao configurar Google Drive: {e}")
    print("[INFO] Usando diretório local como fallback")
    OUTPUT_DIR = COMFYUI_DIR / "output"
    WORKFLOWS_DIR = COMFYUI_DIR / "user"
    LOGS_DIR = COMFYUI_DIR / "logs"
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    WORKFLOWS_DIR.mkdir(parents=True, exist_ok=True)
    LOGS_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
# ComfyUI + custom nodes (com output_dir no Drive)
from comfyui_setup import setup_comfyui

CUSTOM_NODES = [
    "ltdrdata/ComfyUI-Manager",
    "cubiq/ComfyUI_essentials",
]

setup_comfyui(
    comfyui_dir=COMFYUI_DIR,
    models_dir=MODELS_DIR,
    custom_nodes=CUSTOM_NODES,
    output_dir=OUTPUT_DIR,
    drive_base=DRIVE_BASE,
)

In [ ]:
import sys
from pathlib import Path
import subprocess

SCRIPTS_DIR = Path("/kaggle/working/scripts")
sys.path.insert(0, str(SCRIPTS_DIR))
DATASET = "automamermaid/comfydocs"
TARGET_DIR = Path("/kaggle/working/ComfyUI/models")

from kaggle_sync import get_dataset_files, filter_dataset_files, sync_dataset_to_local, MODEL_CATEGORIES

def choose_dataset_files(dataset=DATASET, preselected_categories=None):
    """Abre uma caixa de seleção no notebook para escolher os arquivos exatos."""
    files = get_dataset_files(dataset)
    if not files:
        raise RuntimeError("O Dataset não possui arquivos ou não pôde ser listado.")

    candidates = filter_dataset_files(files, categories=preselected_categories)
    if not candidates:
        candidates = files

    try:
        import ipywidgets as widgets
        from IPython.display import display, clear_output

        options = [(f"{Path(f).name}  [{f}]", f) for f in candidates]
        selector = widgets.SelectMultiple(
            options=options,
            rows=min(18, max(5, len(options))),
            description="Modelos:",
            layout=widgets.Layout(width="100%", height="420px"),
        )
        select_all = widgets.Button(description="Selecionar todos")
        clear_all = widgets.Button(description="Limpar")
        confirm = widgets.Button(description="Confirmar seleção", button_style="success")
        output = widgets.Output()

        def all_click(_): selector.value = tuple(candidates)
        def clear_click(_): selector.value = tuple()
        def confirm_click(_):
            with output:
                clear_output(wait=True)
                chosen = list(selector.value)
                if not chosen:
                    print("Nenhum arquivo selecionado.")
                else:
                    print("Selecionados:")
                    for item in chosen: print("  -", item)
                    print(f"Total: {len(chosen)} arquivo(s)")

        select_all.on_click(all_click)
        clear_all.on_click(clear_click)
        confirm.on_click(confirm_click)
        display(widgets.HTML(f"<b>{dataset}</b> · {len(files)} arquivo(s) disponíveis · {len(candidates)} candidato(s)"))
        display(widgets.HBox([select_all, clear_all, confirm]))
        display(selector, output)

        # A execução da célula continua após a seleção. Aguarde a interface e rode a próxima célula.
        return selector
    except ImportError:
        print("ipywidgets não disponível. Use a lista abaixo e informe os paths manualmente.")
        for i, f in enumerate(candidates, 1): print(f"{i:03d}: {f}")
        raw = input("Números separados por vírgula: ").strip()
        indexes = [int(x)-1 for x in raw.split(',') if x.strip().isdigit()]
        return [candidates[i] for i in indexes if 0 <= i < len(candidates)]


In [ ]:
# ESCOLHA MANUAL DOS MODELOS
# Marque apenas o que será copiado para o SSD local nesta execução.
CATEGORIES = ["checkpoints", "diffusion_models", "loras", "vae", "text_encoders", "clip", "controlnet", "upscale_models", "video_models", "embeddings"]
selector = choose_dataset_files(DATASET, preselected_categories=CATEGORIES)

In [ ]:
if hasattr(selector, "value"):
    SELECTED_FILES = list(selector.value)
else:
    SELECTED_FILES = list(selector)

if not SELECTED_FILES:
    raise ValueError("Nenhum modelo selecionado. Execute novamente a célula de seleção.")

stats = sync_dataset_to_local(
    dataset=DATASET,
    target_dir=MODELS_DIR,
    selected_files=SELECTED_FILES,
    force=False,
)
print(json.dumps(stats, indent=2, ensure_ascii=False))

In [ ]:
# Inicia ComfyUI em background com output no Drive e valida a API.
from comfyui_setup import start_comfyui, health_check

COMFYUI_PORT = 8188
proc = start_comfyui(
    comfyui_dir=COMFYUI_DIR,
    host="0.0.0.0",
    port=COMFYUI_PORT,
    output_dir=OUTPUT_DIR,
)

if not health_check("127.0.0.1", COMFYUI_PORT, timeout=90):
    raise RuntimeError("ComfyUI iniciou, mas não respondeu ao health check.")

print(f"ComfyUI OK em http://127.0.0.1:{COMFYUI_PORT}")
print(f"Outputs salvos em: {OUTPUT_DIR}")
print("PID:", proc.pid)

## Próximo passo

Se todas as células passarem, a arquitetura Kaggle está validada até o health check. A geração de uma imagem real deve ser o teste seguinte, usando um workflow mínimo e somente os modelos que você selecionou.

**Outputs vão direto para o Google Drive** (pasta `Automa/ComfyUI/outputs/`). Para sincronizar workflows salvos ou puxar do Drive, execute `09_sync_outputs.ipynb`.